# aşama 3, tekrar yönü denemeleri (kaggle)

colab'da çalıştırdığım üç tekrar yönü denemesinin ham cevapları oturum sıfırlanınca kayboldu. burada aynı denemeleri baştan çalıştırıyorum.
düzenlenmemiş tabanı da aynı ortamda yeniden alıyorum, böylece dört satır aynı kütüphane sürümüyle karşılaştırılıyor.

ayarlar: accelerator **GPU T4 x2**, internet **açık**. "save version > save & run all" ile arka planda çalışıyor. bütün sonuçlar `/kaggle/working` içine yazılıyor ve sürümün output'unda kalıyor.


In [ ]:
!pip install -q -U transformers accelerate datasets math-verify
!git clone -q https://github.com/FurkanKY/modelin-icini-acmak.git /kaggle/working/repo
import sys; sys.path.insert(0, "/kaggle/working/repo/src")
import transformers, torch; print(transformers.__version__, torch.cuda.device_count(), "gpu")

In [ ]:
import json, os, time, threading, queue, traceback
import numpy as np, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from stage3_ablate import edit, undo, run, grade, repeat_direction, REPEAT_RUNS

OUT = "/kaggle/working"
NAME = "Qwen/Qwen3.5-0.8B"

def load(dev):
    # her gpu'ya ayrı model ve ayrı tokenizer. iki thread aynı tokenizer'ı kullanınca hata verebiliyor.
    tok = AutoTokenizer.from_pretrained(NAME); tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float16, device_map=dev).eval()
    def build_prompt(q):
        msg = [{"role": "user", "content": q + "\n\nPlease reason step by step, and put your final answer within \\boxed{}."}]
        return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    return model, tok, build_prompt, tok.convert_tokens_to_ids("<|im_end|>")

workers = [load(f"cuda:{i}") for i in range(torch.cuda.device_count())]
model, tok, build_prompt, IM_END = workers[0]
res200 = json.load(open("/kaggle/working/repo/data/stage2_200_raw.json"))
u15 = np.load("/kaggle/working/repo/data/u_layer15.npy")
test = load_dataset("HuggingFaceH4/MATH-500", split="test").select(range(200, 300))

# prompt colab'dakiyle aynı mı, ilk cevabın token'larıyla kontrol
p = tok(build_prompt(res200[0]["problem"]))["input_ids"]
print("prompt token:", len(p)); print(build_prompt("2+2?"))

In [ ]:
# tekrar yönü. 200 cevabın 512. token'dan sonraki kısmı, 8 token'lık tekrar kuralı.
t0 = time.time()
U_rep, n_rep, n_new = repeat_direction(model, tok, build_prompt, res200)
np.save(f"{OUT}/u_rep_all_layers.npy", U_rep)
print(f"{time.time()-t0:.0f} sn, tekrar token {n_rep}, yeni token {n_new} (colab'da 126111 ve 185213)")
print("15. katmanda bitirme yönüyle kosinüs:", round(float(U_rep[15] @ u15), 3), "(colab'da 0.14)")
u_rep = U_rep[15]

In [ ]:
# dört deneme, iki gpu. her gpu kuyruktan bir deneme alıyor, kendi modelini düzenliyor, bitince geri alıyor.
jobs = queue.Queue()
for j in [("base", None)] + REPEAT_RUNS: jobs.put(j)
status = {}

def worker(i):
    m, t, bp, im_end = workers[i]
    while True:
        try: name, kw = jobs.get_nowait()
        except queue.Empty: return
        try:
            sv = edit(m, u_rep, **kw) if kw else []
            try: run(m, t, bp, im_end, test, f"{OUT}/stage3_{name}.json", batch_size=50,
                     log=lambda s, i=i: status.__setitem__(i, s))
            finally: undo(sv)
        except Exception:
            status[f"err_{name}"] = traceback.format_exc()

t0 = time.time()
ths = [threading.Thread(target=worker, args=(i,)) for i in range(len(workers))]
for th in ths: th.start()
while any(th.is_alive() for th in ths):
    time.sleep(300); print(f"{(time.time()-t0)/60:.0f} dk", status, flush=True)
for th in ths: th.join()
print("bitti", f"{(time.time()-t0)/60:.0f} dk", {k: v for k, v in status.items() if str(k).startswith("err")})

In [ ]:
# puanlama ana thread'de. math-verify signal kullanıyor.
R = {}
for name in ["base"] + [n for n, _ in REPEAT_RUNS]:
    R[name] = json.load(open(f"{OUT}/stage3_{name}.json")); g = grade(R[name])
    json.dump(R[name], open(f"{OUT}/stage3_{name}.json", "w"))
    b = R["base"]
    t2f = sum((not x["finished"]) and y["finished"] for x, y in zip(b, R[name]))
    f2t = sum(x["finished"] and not y["finished"] for x, y in zip(b, R[name]))
    print(f"{name:9s} takılan {g['takilan']:3d}  doğru {g['dogru']:3d}  ort.token {int(np.mean([r['n_tok'] for r in R[name]])):5d}  takıldı→bitti {t2f:3d}  bitti→takıldı {f2t:3d}")